In [1]:
import shapely.geometry as geometry
from shapely.ops import linemerge, unary_union, polygonize
from shapely.geometry import mapping

import networkx as nx
import pandas as pd
import numpy as np
import pickle

import matplotlib.pyplot as plt

from haversine import haversine

from convenient_pickle import *

import os
import time
import gc
import geojson

safe_cwd = os.getcwd()

%config InteractiveShell.cache_size = 0

In [2]:
def load_pickles(prefix):
    G = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_graph.pkl')
    total_result_nodes = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_total_result_nodes.pkl')
    total_result_ways = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_total_result_ways.pkl')
    used_bboxes = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_used_bboxes.pkl')
    return G, total_result_nodes, total_result_ways, used_bboxes

def lite_load_pickles(prefix):
    G = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_graph.pkl')
    total_result_ways = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_total_result_ways.pkl')
    return G, total_result_ways

def make_ways_lighter(total_result_ways) :
    ways_dict = dict()
    ways_dict['ways'] = dict()
    ways_dict['nodes'] = dict()
    for way in total_result_ways: 
        way_id = way.id
        node_ids = []
        for node in way.nodes: 
            node_ids.append(node.id)
            ways_dict[node.id] = {'lat': float(node.lat),'lon': float(node.lon)}
        ways_dict['ways'][way_id] = node_ids
    return ways_dict
            

In [3]:
def get_edge_distance(node1, node2): 
    lat1 = float(node1['lat'])
    lon1 = float(node1['lon'])
    lat2 = float(node2['lat'])
    lon2 = float(node2['lon'])

    return haversine((lat1,lon1), (lat2, lon2))


In [4]:
def convert_to_borders(ways):
    lss = [] 
    
    for ii_w,way in enumerate(ways):
        ls_coords = []
    
        for node in way.nodes:
            ls_coords.append((node.lon,node.lat)) 
    
        lss.append(geometry.LineString(ls_coords))
    
    
    merged = linemerge([*lss]) 
    borders = unary_union(merged) # linestrings to a MultiLineString
    polygons = list(polygonize(borders))
    return merged, borders, polygons

def convert_to_dispersed_borders(ways):
    lss = [] 
    
    for ii_w,way in enumerate(ways):
        ls_coords = []
    
        for node in way.nodes:
            ls_coords.append((node.lon,node.lat)) 
    
        lss.append(geometry.LineString(ls_coords))
    
    
    merged = linemerge([*lss]) 
    return merged
    
#See how many ways each node belongs to
def collect_overlap(ways):
    outdict = dict()
    for way in ways: 
        way_nodes = way.nodes
        for node in way_nodes: 
            if node.id not in outdict.keys(): 
                outdict[node.id] = 1
            else: 
                outdict[node.id] += 1
    outdict = [(i, outdict[i]) for i in outdict.keys()]
    outdict = sorted(outdict, key = lambda x: -x[1])
    return outdict

def make_graph(ways):
    G = nx.Graph()
    
    for way in ways: 
        for node_dex in range(len(way.nodes)): 
            node = way.nodes[node_dex]
            node_id = node.id
            if G.has_node(node_id) == False:
                G.add_node(node_id)
            if node_dex > 0: 
                source = {'lat':node.lat, 'lon':node.lon}
                target = {'lat':way.nodes[node_dex-1].lat, 'lon':way.nodes[node_dex-1].lon}
                edge_distance = get_edge_distance(source, target)
                G.add_edge(node_id,way.nodes[node_dex-1].id,weight=edge_distance)
                
    return G

def make_new_lcc(G, ways):
    cclist = sorted([i for i in nx.connected_components(G)], key = lambda x: -len(x))
    lcc = cclist[0]
    lcc_ways = []
    for way in ways: 
        for node in way.nodes: 
            if node.id in lcc: 
                lcc_ways.append(way)
                break
    lcc_node_set = set()
    for way in lcc_ways:
        for node in way.nodes:
            lcc_node_set.add(node.id)
    return pd.Series(lcc_ways)

# Serialize your merged network to GeoJSON
def quick_map(lcc_merged):
    bike_geojson = mapping(lcc_merged)
    
    # Illinois center
    m = folium.Map(location=[40.0, -89.2], zoom_start=6, tiles='CartoDB positron')
    
    # Illinois outline
    folium.GeoJson(
        "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
        name="Illinois",
        style_function=lambda f: {
            'fillColor': '#E6F1FB',
            'color': '#185FA5',
            'weight': 1.5,
            'fillOpacity': 0.25
        } if f['properties']['name'] == 'Illinois' else {
            'fillOpacity': 0,
            'color': 'none',
            'weight': 0
        }
    ).add_to(m)
    
    # Bike/pedestrian network
    folium.GeoJson(
        bike_geojson,
        name="Bike network",
        style_function=lambda f: {
            'color': '#1D9E75',
            'weight': 2,
            'opacity': 0.85
        }
    ).add_to(m)
    
    folium.Rectangle(
        bounds=[[bbox[0], bbox[1]], [bbox[2], bbox[3]]],
        color='#E24B4A',
        weight=2,
        fill=False
    ).add_to(m)
    
    # new_bbox = get_new_bbox(bbox, interval, 'northeast')
    
    # folium.Rectangle(
    #     bounds=[[new_bbox[0], new_bbox[1]], [new_bbox[2], new_bbox[3]]],
    #     color='#E24B4A',
    #     weight=2,
    #     fill=False
    # ).add_to(m)
    
    
    # Zoom to the network
    bounds = lcc_merged.bounds  # (minx, miny, maxx, maxy)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    
    folium.LayerControl().add_to(m)
    return m

def break_down_into_nodes(ways): 
    node_list = []
    for way in ways: 
        node_list += way.nodes
    return node_list

def get_node_ids(nodelist):
    id_list = []
    for node in nodelist: 
        id_list.append(node.id)
    return id_list

def get_way_node_ids(ways): 
    node_list = break_down_into_nodes(ways)
    id_list = get_node_ids(node_list)
    return id_list

def average_degree(subgraph): 
    average = 0
    for node in subgraph.nodes: 
        average += subgraph.degree(node)
    return average/len(subgraph.nodes)

def get_node_latlon_dict(ways): 
    outdict = dict()
    nodelist = break_down_into_nodes(ways)
    for node in nodelist: 
        outdict[node.id] = {'lat': node.lat, 'lon': node.lon}
    return outdict

def get_edge_distance(node1, node2): 
    lat1 = float(node1['lat'])
    lon1 = float(node1['lon'])
    lat2 = float(node2['lat'])
    lon2 = float(node2['lon'])

    return haversine((lat1,lon1), (lat2, lon2))

def get_graph_distances(use_graph, ways):
    node_latlon_dict = get_node_latlon_dict(ways)
    edges = use_graph.edges
    out_dist = 0
    for edge in edges: 
        node1 = node_latlon_dict[edge[0]]
        node2 = node_latlon_dict[edge[1]]
        out_dist += get_edge_distance(node1, node2)
    return out_dist

def get_degree_distribution(use_graph): 
    outdict = dict()
    for node in use_graph.nodes: 
        degree = use_graph.degree(node)
        if degree not in outdict.keys(): 
            outdict[degree] = 1
        else: 
            outdict[degree] += 1
    return outdict

def find_intersections(outdict): 
    intersections = 0
    for key in outdict.keys(): 
        if key >= 3: 
            intersections += outdict[key]
    return intersections

# Removes all nodes that have a degree of 2, connects intersections/dead ends directly to each other
def strip_the_paths(use_ingraph):
    ingraph = use_ingraph.copy()
    nodes = list(ingraph.nodes)
    for node in nodes: 
        degree = ingraph.degree(node)
        if degree == 2: 
            neighbors = [i for i in ingraph.neighbors(node)]
            ingraph.remove_node(node)
            ingraph.add_edge(neighbors[0], neighbors[1])
    return ingraph

def get_community_box_area(inshape): 
    box = inshape.bounds
    topleft = (box[0],box[1])
    topright = (box[0],box[3])
    bottomleft = (box[2],box[1])
    bottomright = (box[2],box[3])
    polygon = geometry.box(box[0], box[1], box[2], box[3])
    return area(mapping(polygon))/1000000

def get_community_hull_area(inshape): 
    return area(mapping(inshape.convex_hull))/1000000

def get_edge_distance(node1, node2): 
    lat1 = float(node1['lat'])
    lon1 = float(node1['lon'])
    lat2 = float(node2['lat'])
    lon2 = float(node2['lon'])

    return haversine((lat1,lon1), (lat2, lon2))

In [5]:

#load in data
prefix = '06_26_2026'
#total_result_ways = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_total_result_ways.pkl')
G, total_result_ways = lite_load_pickles(prefix)

# Get lats and lons for all nodes
node_dict = dict() 

for way in total_result_ways: 
    for node in way.nodes: 
        node_dict[node.id] = {'lat': node.lat, 'lon': node.lon}

# Go through all nodes in node_dict, get neighbors in graph, find distance between unique pairs of neighbors, add to list

distance_list = []
distance_pairs = set()

for node in node_dict.keys(): 
    neighbors = [i for i in G.neighbors(node)]
    for neighbor in neighbors: 
        sorted_pair = tuple(sorted([node, neighbor]))
        if sorted_pair not in distance_pairs: 
            edge_distance = get_edge_distance(node_dict[neighbor], node_dict[node])
            distance_list.append(edge_distance)
            distance_pairs.add(sorted_pair)

#allow to compare

median = np.median(np.array(distance_list))
dump_pickle('misc_info','inter_node_median', median)
#input('bring the system monitor over here and compare')


%xdel node_dict
gc.collect()
%xdel distance_pairs
gc.collect()




0

In [6]:
prefix_bike = '06_26_2026_just_bikes'
total_result_ways_bike = load_pickle('pickle_folder/'+prefix_bike + '/' + prefix_bike+'_total_result_ways.pkl')
outlist = []
for way in total_result_ways_bike: 
    for node in way.nodes: 
        if node.id in G.nodes: 
            outlist.append(way)

In [7]:
test = convert_to_borders(total_result_ways_bike)

In [8]:
out_geojson = mapping(test[0])

In [9]:
with open(f'geojsons/bike_lcc_infrastructure.geojson', 'w') as file:
    geojson.dump(out_geojson, file)

In [10]:
print([i for i in total_result_ways_bike if i.id==1069101519])

[<overpy.Way id=1069101519 nodes=[12352236488, 12352236486, 339214357]>]


In [11]:
G